In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore') # Wyłączenie ostrzeżeń o braku zbieżności



In [2]:
# === PATHS SETUP ===
SCRIPT_PATH = os.getcwd()
DATA_DIR = os.path.abspath(os.path.join(SCRIPT_PATH, '..', '..', 'data', 'preprocessed', 'analysis_data'))
MODELS_DIR = os.path.abspath(os.path.join(SCRIPT_PATH, '..', '..', 'models', 'bayesian'))
RESULTS_DIR = os.path.abspath(os.path.join(SCRIPT_PATH, '..', '..', 'results', 'bayesian'))
FEATURES_NAMES_DIR = os.path.abspath(os.path.join(DATA_DIR,'feature_names.pkl'))
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
# === DATA LOADING ===
print("Loading data...")
train_path = os.path.join(DATA_DIR, 'train.csv')
test_path = os.path.join(DATA_DIR, 'test.csv')

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

target_col = 'Fault_Condition'
feature_cols = joblib.load(FEATURES_NAMES_DIR)

X_train = train_data[feature_cols]
y_train_raw = train_data[target_col]
X_test = test_data[feature_cols]
y_test_raw = test_data[target_col]

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

Loading data...


In [4]:
X_full_cv = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
y_full_cv = np.concatenate([y_train, y_test])

# Parameters for Bayesian aproximation
param_grid_mlr = {
    'C': np.logspace(-3, 3, 10),
    'solver': ['lbfgs', 'saga'],
    'max_iter': [2000]
}

# Softmax with l2 regularization
bayesian_map_model = LogisticRegression(multi_class='multinomial', penalty='l2', random_state=42)

In [5]:
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

clf_map = GridSearchCV(
    estimator=bayesian_map_model,
    param_grid=param_grid_mlr,
    cv=inner_cv,
    scoring='accuracy',
    n_jobs=-1
)
scoring_metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

# Uruchomienie pętli zewnętrznej
nested_cv_results = cross_validate(
    clf_map,
    X_full_cv,
    y_full_cv,
    cv=outer_cv,
    scoring=scoring_metrics,
    n_jobs=-1
)

In [7]:
acc_mean, acc_std = nested_cv_results['test_accuracy'].mean() * 100, nested_cv_results['test_accuracy'].std() * 100
prec_mean, prec_std = nested_cv_results['test_precision_macro'].mean() * 100, nested_cv_results['test_precision_macro'].std() * 100
rec_mean, rec_std = nested_cv_results['test_recall_macro'].mean() * 100, nested_cv_results['test_recall_macro'].std() * 100
f1_mean, f1_std = nested_cv_results['test_f1_macro'].mean() * 100, nested_cv_results['test_f1_macro'].std() * 100

print("\n=== NESTED CV evaluation for Bayesian Softmax ===")
print(f"Accuracy:  {acc_mean:.1f}% ± {acc_std:.1f}%")
print(f"Precision: {prec_mean:.1f}% ± {prec_std:.1f}%")
print(f"Recall:    {rec_mean:.1f}% ± {rec_std:.1f}%")
print(f"F1-Score:  {f1_mean:.1f}% ± {f1_std:.1f}%")


=== NESTED CV evaluation for Bayesian Softmax ===
Accuracy:  89.1% ± 3.6%
Precision: 90.0% ± 3.5%
Recall:    89.2% ± 3.7%
F1-Score:  89.0% ± 3.7%
